# Colab T4 벤치마크 — 노트북(RTX 5060)과 동일 조건 비교

**준비**: 로컬에서 `python tools/make_colab_bundle.py` 실행 → `colab/colab_bundle.zip`을 Google Drive `MyDrive/microstructure/`에 업로드
런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 위에서부터 실행.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/work && cd /content/work && unzip -q -o /content/drive/MyDrive/microstructure/colab_bundle.zip && ls

In [ ]:
%cd /content/work
!pip -q install timm PyWavelets
import torch, timm; print(torch.__version__, timm.__version__, torch.cuda.get_device_name(0))

## 1) 처리량 벤치마크 (T4 최적 설정: fp16 + channels_last)

In [ ]:
!python bench/gpu_benchmark.py --dtype fp16 --channels-last --out bench/result_T4_fp16_cl.csv

In [ ]:
!python bench/gpu_benchmark.py --dtype fp16 --out bench/result_T4_fp16_nchw.csv

## 2) 실제 학습 시간 비교 (노트북과 동일 실험: resnet18, fold 0, 25 epoch)

In [ ]:
!python src/train_cnn.py --model resnet18.a1_in1k --folds 0 --epochs 25 --lr 1e-3 --dtype fp16 --channels-last --exp t4_resnet18_check

## 3) 결과 저장 (Drive로 복사 → 로컬 `bench/`에 넣어 비교)

In [ ]:
!mkdir -p /content/drive/MyDrive/microstructure/results && cp -r bench/result_T4_*.csv outputs/cnn /content/drive/MyDrive/microstructure/results/ && ls /content/drive/MyDrive/microstructure/results